# Dielectric Tensor (Optical Dielectric Function)

Calculate the frequency-dependent optical dielectric function ε(ω) of a material using a DFT workflow on the Mat3ra platform (Quantum ESPRESSO: `pw.x` SCF/NSCF + `epsilon.x` independent-particle dielectric response).

<h2 style="color:green">Usage</h2>

1. Set material and calculation parameters in cells 1.2 and 1.3 below (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for material, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create material: materials are read from the `../uploads` folder — place files there manually or run a material creation notebook first. If the material is not found by name, Standata is used as a fallback. The material is then saved to the platform.
1. Configure workflow: select application, load dielectric tensor workflow from Standata, optionally add relaxation, set model and computational parameters, and save the workflow.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with material and workflow configuration: assemble the job from material, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the dielectric tensor (real and imaginary parts of ε(ω)).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # small primitive cell (FCC, mp-149); NC pseudopotential available

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "dielectric_tensor.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Dielectric Function"

# Model parameters
MODEL_SUBTYPE = "gga"  # "gga" or "lda"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific dielectric tensor parameters

In [ ]:
# Method parameters
# NOTE: epsilon.x (used to compute the dielectric function) does NOT support
# ultrasoft or PAW pseudopotentials -- norm-conserving ("nc") is required.
PSEUDOPOTENTIAL_TYPE = "nc"  # "nc" (norm-conserving) -- required; "us"/"paw" will fail
FUNCTIONAL = "pbe"           # for gga: "pbe", "pbesol"; for lda: "pz"

# Relax the structure before computing the dielectric response. Recommended for
# any material not already at its equilibrium geometry (e.g. Graphene / 2D).
ADD_RELAXATION = False

# Electronic k-grids; if None, the workflow default is used
RELAXATION_KGRID = None  # e.g. [8, 8, 8]  (used only when ADD_RELAXATION)
SCF_KGRID = None         # e.g. [8, 8, 8] (3D) or [12, 12, 1] (2D)
# The NSCF step recomputes the full (non-symmetry-reduced) k-mesh that epsilon.x
# integrates over; for a converged dielectric function it should be at least as
# dense as, and typically denser than, the SCF grid (e.g. [16, 16, 16]).
NSCF_KGRID = None       # e.g. [16, 16, 16] (3D) or [24, 24, 1] (2D)

# Energy cutoffs (raise for harder cases -- e.g. 60 / 480 for Graphene)
ECUTWFC = 40
ECUTRHO = 200

# epsilon.x energy grid (all values in eV)
ENERGY_GRID_MIN = 0.0     # wmin
ENERGY_GRID_MAX = 15.0    # wmax
ENERGY_GRID_POINTS = 500  # nw
INTERSMEAR = 0.2   # Lorentzian broadening between bands (eV)
INTRASMEAR = 0.0   # Drude-like intraband broadening (eV); set > 0 for metals
ENERGY_GRID_SHIFT = 0.0   # rigid shift of the energy grid (eV)

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

material.basis.set_labels_from_list([])
saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load workflow from Standata and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Add relaxation (optional)

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()
    visualize_workflow(workflow)

### 4.4. Set Model and its parameters (physics)

In [ ]:
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.mode import ModelFactory

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    subworkflow.model = model

visualize_workflow(workflow)

### 4.5. Modify Method (computational parameters): k-grids, cutoffs, and energy grid

In [ ]:
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input

dielectric_subworkflow = workflow.subworkflows[1 if ADD_RELAXATION else 0]

# Relaxation electronic k-grid
if RELAXATION_KGRID is not None and ADD_RELAXATION:
    unit = workflow.subworkflows[0].get_unit_by_name(name_regex="relax")
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=RELAXATION_KGRID, isEdited=True).get_context_item_data())
        workflow.subworkflows[0].set_unit(unit)

# SCF electronic k-grid (pw.x)
if SCF_KGRID is not None:
    unit = dielectric_subworkflow.get_unit_by_name(name="pw_scf")
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).get_context_item_data())
        dielectric_subworkflow.set_unit(unit)

# NSCF electronic k-grid (pw.x) -- the grid epsilon.x integrates over
if NSCF_KGRID is not None:
    unit = dielectric_subworkflow.get_unit_by_name(name="pw_nscf")
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=NSCF_KGRID, isEdited=True).get_context_item_data())
        dielectric_subworkflow.set_unit(unit)

# Energy cutoffs on every pw.x unit
if ECUTWFC is not None:
    cutoffs_context = PlanewaveCutoffsContextProvider(
        wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True
    ).get_context_item_data()
    for unit_name in ["pw_relax", "pw_vc-relax", "pw_scf", "pw_nscf"]:
        for swf in workflow.subworkflows:
            unit = swf.get_unit_by_name(name=unit_name)
            if unit:
                unit.add_context(cutoffs_context)
                swf.set_unit(unit)

# epsilon.x energy grid
patch_workflow_qe_input(
    workflow,
    {
        "energy_grid": {
            "wmin": ENERGY_GRID_MIN,
            "wmax": ENERGY_GRID_MAX,
            "nw": ENERGY_GRID_POINTS,
            "intersmear": INTERSMEAR,
            "intrasmear": INTRASMEAR,
            "shift": ENERGY_GRID_SHIFT,
        }
    },
    unit_names=["Compute dielectric function"],
)

### 4.6. Preview final workflow

In [ ]:
visualize_workflow(workflow)

### 4.7. Save workflow to collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.ui import display_JSON

job_name = f"{MY_WORKFLOW_NAME} {saved_material.formula} {timestamp}"
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print(f"✅ Job created: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Dielectric Tensor

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

dielectric_tensor_data = get_properties_for_job(
    client, job_id, property_name=PropertyName.non_scalar.dielectric_tensor.value
)
visualize_properties(dielectric_tensor_data, title="Dielectric Tensor", extra_config={"material": material.to_dict()})